# 📘 Clase 19: Regresión lineal y múltiple

**Idea central:** La regresión lineal encuentra la línea que mejor resume la relación entre una variable y lo que queremos predecir.

En esta clase aprenderemos a predecir números continuos (no categorías), a leer coeficientes e interceptos, y a evaluar qué tan bien explica el modelo los datos usando R².

## 📦 Celda 1 — Importar librerías y cargar datos

In [ ]:
# Qué hace: importar las librerías necesarias y cargar el dataset de ventas.
# Para qué sirve: prepara el entorno de trabajo para el análisis de regresión.
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

df = pd.read_csv("../../datasets/ventas_tienda.csv")
print("Filas y columnas:", df.shape)
df.head()

## 📊 Celda 2 — Exploración visual: scatter plot

Antes de entrenar cualquier modelo, miramos los datos. ¿Parece que hay una relación lineal?

In [ ]:
# Qué hace: graficar unidades vs total_neto para ver si hay tendencia lineal.
# Para qué sirve: la intuición visual es el primer paso antes de cualquier modelo.
plt.figure(figsize=(8, 5))
plt.scatter(df["unidades"], df["total_neto"], alpha=0.5, color="steelblue")
plt.xlabel("Unidades vendidas")
plt.ylabel("Total neto ($)")
plt.title("¿A más unidades, mayor total neto?")
plt.tight_layout()
plt.show()

## 🔢 Celda 3 — Regresión lineal simple

Entrenamos un modelo usando solo `unidades` para predecir `total_neto`.

In [ ]:
# Qué hace: entrenar LinearRegression con una sola variable predictora.
# Para qué sirve: introduce la estructura básica del flujo de regresión en sklearn.
X = df[["unidades"]]   # doble corchete → matriz 2D (sklearn lo requiere)
y = df["total_neto"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo_simple = LinearRegression()
modelo_simple.fit(X_train, y_train)

print("Intercepto (costo fijo):", round(modelo_simple.intercept_, 2))
print("Coeficiente (ganancia por unidad):", round(modelo_simple.coef_[0], 2))
print("R² en test:", round(modelo_simple.score(X_test, y_test), 3))

## 📈 Celda 4 — Visualizar la línea de regresión

Graficamos la línea que el modelo aprendió sobre los datos reales.

In [ ]:
# Qué hace: superponer la línea de regresión sobre el scatter plot.
# Para qué sirve: ver visualmente qué tan bien resume la línea la nube de puntos.
x_linea = np.linspace(df["unidades"].min(), df["unidades"].max(), 100).reshape(-1, 1)
y_linea = modelo_simple.predict(x_linea)

plt.figure(figsize=(8, 5))
plt.scatter(df["unidades"], df["total_neto"], alpha=0.4, color="steelblue", label="Datos reales")
plt.plot(x_linea, y_linea, color="red", linewidth=2, label="Línea de regresión")
plt.xlabel("Unidades")
plt.ylabel("Total neto ($)")
plt.title("Regresión lineal simple")
plt.legend()
plt.tight_layout()
plt.show()

## ➕ Celda 5 — Regresión múltiple: añadir más variables

Agregamos `descuento_pct` como segunda variable. ¿Mejora el modelo?

In [ ]:
# Qué hace: entrenar un modelo con dos variables predictoras.
# Para qué sirve: mostrar cómo agregar features puede mejorar la capacidad predictiva.
X_multi = df[["unidades", "descuento_pct"]]
y = df["total_neto"]

X_m_train, X_m_test, y_m_train, y_m_test = train_test_split(
    X_multi, y, test_size=0.2, random_state=42
)

modelo_multi = LinearRegression()
modelo_multi.fit(X_m_train, y_m_train)

print("Intercepto:", round(modelo_multi.intercept_, 2))
for nombre, coef in zip(X_multi.columns, modelo_multi.coef_):
    print(f"  Coeficiente '{nombre}':", round(coef, 3))
print("R² múltiple en test:", round(modelo_multi.score(X_m_test, y_m_test), 3))

## 🔮 Celda 6 — Hacer predicciones con el modelo

Usamos `model.predict()` para estimar ventas en escenarios hipotéticos.

In [ ]:
# Qué hace: predecir total_neto para ventas hipotéticas.
# Para qué sirve: practicar el uso final del modelo (inferencia sobre nuevos datos).
nuevas_ventas = pd.DataFrame({
    "unidades": [10, 50, 100],
    "descuento_pct": [0, 10, 20]
})

predicciones = modelo_multi.predict(nuevas_ventas)

for i, pred in enumerate(predicciones):
    u = nuevas_ventas.iloc[i]["unidades"]
    d = nuevas_ventas.iloc[i]["descuento_pct"]
    print(f"  {u} unidades con {d}% descuento → total_neto predicho: ${pred:,.2f}")

## 📉 Celda 7 — Residuos: dónde se equivoca el modelo

In [ ]:
# Qué hace: calcular y graficar los residuos (error real - predicho).
# Para qué sirve: detectar si el modelo tiene sesgo sistemático en alguna zona.
y_pred_test = modelo_multi.predict(X_m_test)
residuos = y_m_test.values - y_pred_test

plt.figure(figsize=(8, 5))
plt.scatter(y_pred_test, residuos, alpha=0.5, color="darkorange")
plt.axhline(0, color="black", linestyle="--", linewidth=1.5)
plt.xlabel("Valores predichos")
plt.ylabel("Residuos (real − predicho)")
plt.title("Gráfico de residuos — modelo múltiple")
plt.tight_layout()
plt.show()

rmse = np.sqrt(mean_squared_error(y_m_test, y_pred_test))
print(f"RMSE (error promedio en la misma unidad que y): ${rmse:,.2f}")

## 🎓 Celda 8 — Dataset de estudiantes: predecir nota

In [ ]:
# Qué hace: aplicar regresión a otro dataset (estudiantes) para generalizar el aprendizaje.
# Para qué sirve: demostrar que el mismo flujo funciona con cualquier problema numérico.
est = pd.read_csv("../../datasets/estudiantes.csv")

X_est = est[["asistencia_pct", "evaluacion_pandas"]]
y_est = est["evaluacion_python"]

X_e_train, X_e_test, y_e_train, y_e_test = train_test_split(
    X_est, y_est, test_size=0.2, random_state=42
)

modelo_est = LinearRegression()
modelo_est.fit(X_e_train, y_e_train)

print("R² (nota Python desde asistencia + nota Pandas):", round(modelo_est.score(X_e_test, y_e_test), 3))
for nombre, coef in zip(X_est.columns, modelo_est.coef_):
    print(f"  '{nombre}': {round(coef, 3)}")

## ⚖️ Celda 9 — Comparar R²: simple vs múltiple

In [ ]:
# Qué hace: mostrar en una tabla comparativa los R² de los modelos entrenados.
# Para qué sirve: visualizar de forma clara el impacto de agregar más variables.
resultados = pd.DataFrame({
    "Modelo": ["Regresión simple (ventas)", "Regresión múltiple (ventas)", "Regresión múltiple (estudiantes)"],
    "R²": [
        round(modelo_simple.score(X_test, y_test), 3),
        round(modelo_multi.score(X_m_test, y_m_test), 3),
        round(modelo_est.score(X_e_test, y_e_test), 3)
    ]
})
print(resultados.to_string(index=False))

## ✅ Celda 10 — Cuándo usar regresión vs clasificación

In [ ]:
# Qué hace: imprimir una tabla de decisión rápida.
# Para qué sirve: consolidar el criterio para elegir el tipo de modelo correcto.
decision = pd.DataFrame({
    "Pregunta": [
        "¿Cuánto venderemos?",
        "¿Cuál será la nota final?",
        "¿Aprobará el estudiante?",
        "¿Es fraude o no?",
        "¿Cuánto costará la casa?"
    ],
    "Tipo de salida": ["Número", "Número", "Categoría", "Categoría", "Número"],
    "Modelo indicado": ["Regresión", "Regresión", "Clasificación", "Clasificación", "Regresión"]
})
print(decision.to_string(index=False))
print("\n💡 Regla: si la respuesta es un número → regresión. Si es una clase → clasificación.")